# SAM2-UNet v3 — Boundary-Guided Module (BGM)

**Cải tiến:** Thêm BGM vào skip connections (x1/x2/x3) giữa RFB và Decoder.

| Thay đổi | Chi tiết |
|---|---|
| Kiến trúc | BGM shared trên 3 skip connections (64ch) |
| Loss | structure + 0.3×Dice + 0.1×BGM boundary |
| Params thêm | ~37K (rất nhẹ) |
| Encoder/Decoder | Giữ nguyên 100% |


## 1. Setup

In [ ]:
!git clone https://github.com/WZH0120/SAM2-UNet.git

Cloning into 'SAM2-UNet'...
remote: Enumerating objects: 316, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 316 (delta 85), reused 58 (delta 58), pack-reused 208 (from 2)
Receiving objects: 100% (316/316), 3.26 MiB | 18.84 MiB/s, done.
Resolving deltas: 100% (125/125), done.


In [ ]:
%cd SAM2-UNet

/content/SAM2-UNet


In [ ]:
!pip install -r requirements.txt

Looking in links: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 6.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 145.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [ ]:
!pip install -q gdown

## 2. Download SAM2 checkpoint

In [ ]:
!mkdir -p /content/checkpoints
!wget -q -O /content/checkpoints/sam2_hiera_large.pt \
  https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt
!ls -lh /content/checkpoints

total 857M
-rw-r--r-- 1 root root 857M Jul 28  2024 sam2_hiera_large.pt


## 3. Download dataset

> **Bỏ qua nếu data đã có từ lần chạy trước.**

In [ ]:
!mkdir -p /content/data/Polyp
%cd /content/data/Polyp

!gdown --fuzzy 'https://drive.google.com/file/d/1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb/view?usp=sharing' -O TrainDataset.zip
!gdown --fuzzy 'https://drive.google.com/file/d/1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao/view?usp=sharing' -O TestDataset.zip
!unzip -q TrainDataset.zip
!unzip -q TestDataset.zip

/content/data/Polyp
Downloading...
From (original): https://drive.google.com/uc?id=1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb
From (redirected): https://drive.google.com/uc?id=1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb&confirm=t&uuid=9b0bb5b3-79dc-4329-93ff-99669d9e2ed3
To: /content/data/Polyp/TrainDataset.zip
100% 419M/419M [00:03<00:00, 115MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao
From (redirected): https://drive.google.com/uc?id=1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao&confirm=t&uuid=fec7408e-6ac9-4c9e-81a3-bf956062fb49
To: /content/data/Polyp/TestDataset.zip
100% 343M/343M [00:03<00:00, 96.2MB/s]


In [ ]:
import os
from pathlib import Path

root = Path('/content/data/Polyp')
for p in [
    'TrainDataset/images','TrainDataset/masks',
    'TestDataset/Kvasir/images','TestDataset/Kvasir/masks',
    'TestDataset/CVC-ClinicDB/images','TestDataset/CVC-ClinicDB/masks',
    'TestDataset/CVC-ColonDB/images','TestDataset/CVC-ColonDB/masks',
    'TestDataset/CVC-300/images','TestDataset/CVC-300/masks',
    'TestDataset/ETIS-LaribPolypDB/images','TestDataset/ETIS-LaribPolypDB/masks',
]:
    full = root / p
    status = len(os.listdir(full)) if full.exists() else 'MISSING'
    print(p, '=>', status)

# Fix symlink nếu cần
src = '/content/data/Polyp/TrainDataset/image'
dst = '/content/data/Polyp/TrainDataset/images'
if os.path.exists(src) and not os.path.exists(dst):
    os.symlink(src, dst)
    print('Symlink created: image -> images')

TrainDataset/images => MISSING
TrainDataset/masks => 1450
TestDataset/Kvasir/images => 100
TestDataset/Kvasir/masks => 100
TestDataset/CVC-ClinicDB/images => 62
TestDataset/CVC-ClinicDB/masks => 62
TestDataset/CVC-ColonDB/images => 380
TestDataset/CVC-ColonDB/masks => 380
TestDataset/CVC-300/images => 60
TestDataset/CVC-300/masks => 60
TestDataset/ETIS-LaribPolypDB/images => 196
TestDataset/ETIS-LaribPolypDB/masks => 196
Symlink created: image -> images


## 4. Ghi file model, train, test vào repo

In [ ]:
%%writefile /content/SAM2-UNet/SAM2UNet_bgm.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from sam2.build_sam import build_sam2


# ─── Giữ nguyên từ gốc ────────────────────────────────────────────────────────

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)

    def forward(self, x1, x2):
        x1    = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1    = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                            diffY // 2, diffY - diffY // 2])
        return self.conv(torch.cat([x2, x1], dim=1))


class Adapter(nn.Module):
    def __init__(self, blk) -> None:
        super().__init__()
        self.block = blk
        dim = blk.attn.qkv.in_features
        self.prompt_learn = nn.Sequential(
            nn.Linear(dim, 32), nn.GELU(),
            nn.Linear(32, dim), nn.GELU()
        )

    def forward(self, x):
        return self.block(x + self.prompt_learn(x))


class BasicConv2d(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size,
                 stride=1, padding=0, dilation=1):
        super().__init__()
        self.conv = nn.Conv2d(in_planes, out_planes, kernel_size=kernel_size,
                              stride=stride, padding=padding,
                              dilation=dilation, bias=False)
        self.bn   = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.bn(self.conv(x))


class RFB_modified(nn.Module):
    def __init__(self, in_channel, out_channel):
        super().__init__()
        self.relu    = nn.ReLU(True)
        self.branch0 = nn.Sequential(BasicConv2d(in_channel, out_channel, 1))
        self.branch1 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 3), padding=(0, 1)),
            BasicConv2d(out_channel, out_channel, kernel_size=(3, 1), padding=(1, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=3, dilation=3)
        )
        self.branch2 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 5), padding=(0, 2)),
            BasicConv2d(out_channel, out_channel, kernel_size=(5, 1), padding=(2, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=5, dilation=5)
        )
        self.branch3 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 7), padding=(0, 3)),
            BasicConv2d(out_channel, out_channel, kernel_size=(7, 1), padding=(3, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=7, dilation=7)
        )
        self.conv_cat = BasicConv2d(4 * out_channel, out_channel, 3, padding=1)
        self.conv_res = BasicConv2d(in_channel, out_channel, 1)

    def forward(self, x):
        x_cat = self.conv_cat(torch.cat(
            (self.branch0(x), self.branch1(x), self.branch2(x), self.branch3(x)), 1
        ))
        return self.relu(x_cat + self.conv_res(x))


# ─── NEW: Boundary-Guided Module ──────────────────────────────────────────────

class BGM(nn.Module):
    """
    Boundary-Guided Module — đặt trên mỗi skip connection (64ch).

    Pipeline:
        F  →  [3×3 Conv → BN → ReLU → 1×1 Conv → Sigmoid]  →  boundary map B
        output = F × (1 + B)

    Ý nghĩa:
        - B ≈ 0 ở vùng nội thất  → F × 1   = giữ nguyên
        - B ≈ 1 ở vùng biên       → F × 2   = khuếch đại ×2
        - B cũng được supervise bởi GT boundary trong loss

    Shared weights: dùng 1 BGM instance cho cả 3 skip (x1, x2, x3)
    vì tất cả đều là 64ch sau RFB → học boundary representation thống nhất.
    Params thêm vào: 64×64×9 + 64 + 64×1 = ~37K — rất nhẹ.
    """

    def __init__(self, channels: int = 64):
        super().__init__()
        self.boundary_branch = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, 1, kernel_size=1, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        """
        Args:
            x : (B, 64, H, W)  — RFB output feature
        Returns:
            refined : (B, 64, H, W)  — boundary-enhanced feature
            b_map   : (B,  1, H, W)  — boundary attention map (for aux loss)
        """
        b_map   = self.boundary_branch(x)   # [0, 1]
        refined = x * (1.0 + b_map)
        return refined, b_map


# ─── SAM2UNet với BGM ─────────────────────────────────────────────────────────

class SAM2UNet_BGM(nn.Module):
    def __init__(self, checkpoint_path=None) -> None:
        super().__init__()

        model_cfg = "sam2_hiera_l.yaml"
        model = build_sam2(model_cfg, checkpoint_path) if checkpoint_path else build_sam2(model_cfg)

        # Xoá các phần không cần (giữ nguyên như gốc)
        del model.sam_mask_decoder
        del model.sam_prompt_encoder
        del model.memory_encoder
        del model.memory_attention
        del model.mask_downsample
        del model.obj_ptr_tpos_proj
        del model.obj_ptr_proj
        del model.image_encoder.neck

        self.encoder = model.image_encoder.trunk

        # Freeze encoder, wrap với Adapter (giữ nguyên như gốc)
        for param in self.encoder.parameters():
            param.requires_grad = False
        self.encoder.blocks = nn.Sequential(*[Adapter(b) for b in self.encoder.blocks])

        # RFB (giữ nguyên như gốc)
        self.rfb1 = RFB_modified(144,  64)
        self.rfb2 = RFB_modified(288,  64)
        self.rfb3 = RFB_modified(576,  64)
        self.rfb4 = RFB_modified(1152, 64)

        # NEW: 1 BGM dùng chung cho 3 skip connections
        self.bgm = BGM(channels=64)

        # Decoder (giữ nguyên như gốc)
        self.up1 = Up(128, 64)
        self.up2 = Up(128, 64)
        self.up3 = Up(128, 64)

        # Output heads (giữ nguyên như gốc)
        self.side1 = nn.Conv2d(64, 1, kernel_size=1)
        self.side2 = nn.Conv2d(64, 1, kernel_size=1)
        self.head  = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1, x2, x3, x4 = self.encoder(x)

        # RFB
        x1 = self.rfb1(x1)   # (B, 64, H/4,  W/4)
        x2 = self.rfb2(x2)   # (B, 64, H/8,  W/8)
        x3 = self.rfb3(x3)   # (B, 64, H/16, W/16)
        x4 = self.rfb4(x4)   # (B, 64, H/32, W/32)

        # BGM trên 3 skip connections (shared module)
        x1, b1 = self.bgm(x1)
        x2, b2 = self.bgm(x2)
        x3, b3 = self.bgm(x3)

        # Decoder (y hệt gốc, chỉ x1/x2/x3 đã được refined)
        x    = self.up1(x4, x3)
        out1 = F.interpolate(self.side1(x), scale_factor=16, mode='bilinear')

        x    = self.up2(x, x2)
        out2 = F.interpolate(self.side2(x), scale_factor=8,  mode='bilinear')

        x   = self.up3(x, x1)
        out = F.interpolate(self.head(x),   scale_factor=4,  mode='bilinear')

        # Upsample boundary maps về 352×352 cho loss supervision
        b1 = F.interpolate(b1, scale_factor=4,  mode='bilinear')   # H/4  → H
        b2 = F.interpolate(b2, scale_factor=8,  mode='bilinear')   # H/8  → H
        b3 = F.interpolate(b3, scale_factor=16, mode='bilinear')   # H/16 → H

        return out, out1, out2, b1, b2, b3


if __name__ == "__main__":
    with torch.no_grad():
        model = SAM2UNet_BGM().cuda()
        x = torch.randn(1, 3, 352, 352).cuda()
        out, out1, out2, b1, b2, b3 = model(x)
        print("seg  :", out.shape)
        print("seg1 :", out1.shape)
        print("seg2 :", out2.shape)
        print("b1   :", b1.shape)
        print("b2   :", b2.shape)
        print("b3   :", b3.shape)

        total    = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"\nTotal params    : {total/1e6:.2f}M")
        print(f"Trainable params: {trainable/1e6:.2f}M")


Writing /content/SAM2-UNet/SAM2UNet_bgm.py


In [ ]:
%%writefile /content/SAM2-UNet/train_bgm.py
import os
import argparse
import torch
import torch.optim as opt
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from dataset import FullDataset
from SAM2UNet_bgm import SAM2UNet_BGM

parser = argparse.ArgumentParser("SAM2-UNet v3 — BGM")
parser.add_argument("--hiera_path",       type=str,   required=True)
parser.add_argument("--train_image_path", type=str,   required=True)
parser.add_argument("--train_mask_path",  type=str,   required=True)
parser.add_argument("--save_path",        type=str,   required=True)
parser.add_argument("--epoch",            type=int,   default=20)
parser.add_argument("--lr",               type=float, default=0.001)
parser.add_argument("--batch_size",       type=int,   default=12)
parser.add_argument("--weight_decay",     type=float, default=5e-4)
parser.add_argument("--dice_w",           type=float, default=0.3,
                    help="weight for Dice loss (đã validate ở v2)")
parser.add_argument("--bgm_w",            type=float, default=0.1,
                    help="weight for BGM boundary supervision")
args = parser.parse_args()


# ─── Loss functions ───────────────────────────────────────────────────────────

def structure_loss(pred, mask):
    """Giữ nguyên từ paper gốc."""
    weit = 1 + 5 * torch.abs(
        F.avg_pool2d(mask, kernel_size=31, stride=1, padding=15) - mask
    )
    wbce  = F.binary_cross_entropy_with_logits(pred, mask, reduce='none')
    wbce  = (weit * wbce).sum(dim=(2, 3)) / weit.sum(dim=(2, 3))
    pred_ = torch.sigmoid(pred)
    inter = ((pred_ * mask) * weit).sum(dim=(2, 3))
    union = ((pred_ + mask) * weit).sum(dim=(2, 3))
    wiou  = 1 - (inter + 1) / (union - inter + 1)
    return (wbce + wiou).mean()


def dice_loss(pred, mask, smooth=1.0):
    """Đã validate ở v2: +nhẹ trên Kvasir & CVC-300."""
    pred_ = torch.sigmoid(pred).view(pred.size(0), -1)
    mask_ = mask.view(mask.size(0), -1)
    inter = (pred_ * mask_).sum(dim=1)
    return (1 - (2 * inter + smooth) / (pred_.sum(dim=1) + mask_.sum(dim=1) + smooth)).mean()


def get_boundary_gt(mask, kernel_size=5):
    """
    GT boundary = mask - erode(mask).
    Dùng để supervise boundary maps b1/b2/b3 từ BGM.
    """
    pad    = kernel_size // 2
    eroded = 1.0 - F.max_pool2d(1.0 - mask, kernel_size=kernel_size,
                                 stride=1, padding=pad)
    # Dilate nhẹ để boundary GT rộng hơn (~3px) → ổn định hơn
    dilated = F.max_pool2d(torch.clamp(mask - eroded, min=0.0),
                           kernel_size=3, stride=1, padding=1)
    return dilated


def bgm_boundary_loss(b_map, mask):
    """
    BCE giữa boundary prediction của BGM và GT boundary.
    b_map: đã sigmoid (từ BGM), range [0,1]
    Dùng soft weighting thay vì sparse pixel để ổn định hơn v1.
    """
    boundary_gt = get_boundary_gt(mask)
    # b_map đã là probability (Sigmoid trong BGM) → dùng binary_cross_entropy
    loss = F.binary_cross_entropy(b_map.clamp(1e-6, 1 - 1e-6), boundary_gt,
                                  reduction='mean')
    return loss


def combined_loss(pred, mask, dice_w):
    return structure_loss(pred, mask) + dice_w * dice_loss(pred, mask)


# ─── Training ─────────────────────────────────────────────────────────────────

def main(args):
    dataset    = FullDataset(args.train_image_path, args.train_mask_path, 352, mode='train')
    dataloader = DataLoader(dataset, batch_size=args.batch_size,
                            shuffle=True, num_workers=8)

    device = torch.device("cuda")
    model  = SAM2UNet_BGM(args.hiera_path).to(device)

    optim     = opt.AdamW([{"params": model.parameters(), "initia_lr": args.lr}],
                          lr=args.lr, weight_decay=args.weight_decay)
    scheduler = CosineAnnealingLR(optim, args.epoch, eta_min=1e-7)

    os.makedirs(args.save_path, exist_ok=True)

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total: {total/1e6:.2f}M | Trainable: {trainable/1e6:.2f}M")
    print(f"Loss = structure+{args.dice_w}×Dice (×3 outputs) + {args.bgm_w}×BGM_boundary (×3 maps)")

    for epoch in range(args.epoch):
        for i, batch in enumerate(dataloader):
            x      = batch['image'].to(device)
            target = batch['label'].to(device)

            optim.zero_grad()
            out, out1, out2, b1, b2, b3 = model(x)

            # Segmentation loss trên 3 outputs (y hệt gốc)
            loss_seg = (combined_loss(out,  target, args.dice_w) +
                        combined_loss(out1, target, args.dice_w) +
                        combined_loss(out2, target, args.dice_w))

            # BGM boundary supervision — nhẹ (weight 0.1)
            loss_bgm = (bgm_boundary_loss(b1, target) +
                        bgm_boundary_loss(b2, target) +
                        bgm_boundary_loss(b3, target))

            loss = loss_seg + args.bgm_w * loss_bgm

            loss.backward()
            optim.step()

            if i % 50 == 0:
                print(f"epoch:{epoch+1}-{i+1}: "
                      f"total:{loss.item():.4f} | "
                      f"seg:{loss_seg.item():.4f} | "
                      f"bgm:{loss_bgm.item():.4f}")

        scheduler.step()

        if (epoch + 1) % 5 == 0 or (epoch + 1) == args.epoch:
            ckpt = os.path.join(args.save_path, f'SAM2-UNet-{epoch+1}.pth')
            torch.save(model.state_dict(), ckpt)
            print(f'[Saved] {ckpt}')


if __name__ == "__main__":
    main(args)


Writing /content/SAM2-UNet/train_bgm.py


In [ ]:
%%writefile /content/SAM2-UNet/test_bgm.py
import argparse
import os
import torch
import imageio
import numpy as np
import torch.nn.functional as F
from SAM2UNet_bgm import SAM2UNet_BGM   # đổi từ SAM2UNet
from dataset import TestDataset

parser = argparse.ArgumentParser()
parser.add_argument("--checkpoint",      type=str, required=True)
parser.add_argument("--test_image_path", type=str, required=True)
parser.add_argument("--test_gt_path",    type=str, required=True)
parser.add_argument("--save_path",       type=str, required=True)
args = parser.parse_args()

device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_loader = TestDataset(args.test_image_path, args.test_gt_path, 352)

model = SAM2UNet_BGM().to(device)       # đổi từ SAM2UNet
model.load_state_dict(torch.load(args.checkpoint), strict=True)
model.eval()
model.cuda()

os.makedirs(args.save_path, exist_ok=True)

for i in range(test_loader.size):
    with torch.no_grad():
        image, gt, name = test_loader.load_data()
        gt    = np.asarray(gt, np.float32)
        image = image.to(device)

        res, _, _, _, _, _ = model(image)   # unpack 6 outputs thay vì 3

        res = F.upsample(res, size=gt.shape, mode='bilinear', align_corners=False)
        res = res.sigmoid().data.cpu()
        res = res.numpy().squeeze()
        res = (res - res.min()) / (res.max() - res.min() + 1e-8)
        res = (res * 255).astype(np.uint8)

        print("Saving " + name)
        imageio.imsave(os.path.join(args.save_path, name[:-4] + ".png"), res)

Writing /content/SAM2-UNet/test_bgm.py


In [ ]:
%cd /content/SAM2-UNet
!ls SAM2UNet_bgm.py train_bgm.py test_bgm.py

/content/SAM2-UNet
SAM2UNet_bgm.py  test_bgm.py  train_bgm.py


## 5. Kiểm tra params

In [ ]:
import sys
sys.path.insert(0, '/content/SAM2-UNet')
import torch
from SAM2UNet_bgm import SAM2UNet_BGM

model = SAM2UNet_BGM('/content/checkpoints/sam2_hiera_large.pt')
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total/1e6:.3f}M')
print(f'Trainable params: {trainable/1e6:.3f}M')
print(f'BGM params added: ~{(trainable - 4.38e6)/1e3:.0f}K')

Total params    : 216.456M
Trainable params: 4.307M
BGM params added: ~-73K


## 6. Smoke test (1 epoch) — kiểm tra loss chạy ổn

In [ ]:
%cd /content/SAM2-UNet

!python train_bgm.py \
  --hiera_path /content/checkpoints/sam2_hiera_large.pt \
  --train_image_path /content/data/Polyp/TrainDataset/images/ \
  --train_mask_path  /content/data/Polyp/TrainDataset/masks/ \
  --save_path /content/ckpt/sam2unet_bgm_smoke/ \
  --epoch 1 \
  --batch_size 12

/content/SAM2-UNet
Total: 216.46M | Trainable: 4.31M
Loss = structure+0.3×Dice (×3 outputs) + 0.1×BGM_boundary (×3 maps)
/usr/local/lib/python3.12/dist-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  warnings.warn(warning.format(ret))
epoch:1-1: total:5.4363 | seg:5.2188 | bgm:2.1751
epoch:1-51: total:2.5114 | seg:2.4358 | bgm:0.7561
epoch:1-101: total:2.0371 | seg:1.9945 | bgm:0.4263
[Saved] /content/ckpt/sam2unet_bgm_smoke/SAM2-UNet-1.pth


In [ ]:
# Kiểm tra log smoke test:
# - seg: giảm đều → OK
# - bgm: epoch 1 thường ~0.3-0.5 → sẽ giảm khi train đủ epoch
# - Nếu total dao động nhiều → báo để giảm bgm_w xuống 0.05
!ls -lh /content/ckpt/sam2unet_bgm_smoke/

total 827M
-rw-r--r-- 1 root root 827M May 15 06:46 SAM2-UNet-1.pth


## 7. Full training (20 epochs)

In [ ]:
%cd /content/SAM2-UNet

!python train_bgm.py \
  --hiera_path /content/checkpoints/sam2_hiera_large.pt \
  --train_image_path /content/data/Polyp/TrainDataset/images/ \
  --train_mask_path  /content/data/Polyp/TrainDataset/masks/ \
  --save_path /content/ckpt/sam2unet_bgm/ \
  --epoch 20 \
  --batch_size 12 \
  --lr 0.001 \
  --dice_w 0.3 \
  --bgm_w 0.0

/content/SAM2-UNet
Total: 216.46M | Trainable: 4.31M
Loss = structure+0.3×Dice (×3 outputs) + 0.0×BGM_boundary (×3 maps)
/usr/local/lib/python3.12/dist-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  warnings.warn(warning.format(ret))
epoch:1-1: total:5.3675 | seg:5.3675 | bgm:1.8271
epoch:1-51: total:3.1581 | seg:3.1581 | bgm:2.4876
epoch:1-101: total:2.3822 | seg:2.3822 | bgm:2.9209
epoch:2-1: total:1.7771 | seg:1.7771 | bgm:3.0679
epoch:2-51: total:1.4553 | seg:1.4553 | bgm:3.1830
epoch:2-101: total:1.0554 | seg:1.0554 | bgm:3.2222
epoch:3-1: total:1.1174 | seg:1.1174 | bgm:3.1084
epoch:3-51: total:1.4856 | seg:1.4856 | bgm:3.1577
epoch:3-101: total:1.0322 | seg:1.0322 | bgm:3.0385
epoch:4-1: total:1.0325 | seg:1.0325 | bgm:3.2108
epoch:4-51: total:0.7070 | seg:0.7070 | bgm:3.1594
epoch:4-101: total:0.7739 | seg:0.7739 | bgm:2.9217
epoch:5-1: total:1.2215 | seg:1.2215 | bgm:2.9601
epoch:5-51: to

In [ ]:
!ls -lh /content/ckpt/sam2unet_bgm/

total 3.3G
-rw-r--r-- 1 root root 827M May 15 06:58 SAM2-UNet-10.pth
-rw-r--r-- 1 root root 827M May 15 07:03 SAM2-UNet-15.pth
-rw-r--r-- 1 root root 827M May 15 07:09 SAM2-UNet-20.pth
-rw-r--r-- 1 root root 827M May 15 06:52 SAM2-UNet-5.pth


## 8. Test — inference 5 datasets

In [ ]:
%%bash
cd /content/SAM2-UNet

for DATASET in Kvasir CVC-ClinicDB CVC-ColonDB CVC-300 ETIS-LaribPolypDB
do
  echo '============================'
  echo "Testing $DATASET"
  echo '============================'
  python test_bgm.py \
    --checkpoint /content/ckpt/sam2unet_bgm/SAM2-UNet-20.pth \
    --test_image_path /content/data/Polyp/TestDataset/$DATASET/images/ \
    --test_gt_path    /content/data/Polyp/TestDataset/$DATASET/masks/ \
    --save_path       /content/preds/sam2unet_bgm/$DATASET/
done

Testing Kvasir
Saving cju0u82z3cuma0835wlxrnrjv.png
Saving cju15wdt3zla10801odjiw7sy.png
Saving cju16ach3m1da0993r1dq3sn2.png
Saving cju16whaj0e7n0855q7b6cjkm.png
Saving cju17z0qongpa0993de4boim4.png
Saving cju1amqw6p8pw0993d9gc5crl.png
Saving cju1bm8063nmh07996rsjjemq.png
Saving cju1c3218411b08014g9f6gig.png
Saving cju1cbokpuiw70988j4lq1fpi.png
Saving cju1cj3f0qi5n0993ut8f49rj.png
Saving cju1cqc7n4gpy0855jt246k68.png
Saving cju1ddr6p4k5z08780uuuzit2.png
Saving cju1f8w0t65en0799m9oacq0q.png
Saving cju1h89h6xbnx08352k2790o9.png
Saving cju1hp9i2xu8e0988u2dazk7m.png
Saving cju2hfqnmhisa0993gpleeldd.png
Saving cju2hjrqcvi2j0801bx1i6gxg.png
Saving cju2hos57llxm08359g92p6jj.png
Saving cju2hqt33lmra0988fr5ijv8j.png
Saving cju2lberzkdzm09938cl40pog.png
Saving cju2mh8t6p07008350e01tx2a.png
Saving cju2nnqrqzp580855z8mhzgd6.png
Saving cju2np2k9zi3v079992ypxqkn.png
Saving cju2omjpeqj5a0988pjdlb8l1.png
Saving cju2osuru0ki00855txo0n3uu.png
Saving cju2pag1f0s4r0878h52uq83s.png
Saving cju2rga4psq9n098

/content/SAM2-UNet/test_bgm.py:35: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  res = F.upsample(res, size=gt.shape, mode='bilinear', align_corners=False)
/content/SAM2-UNet/test_bgm.py:35: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  res = F.upsample(res, size=gt.shape, mode='bilinear', align_corners=False)
/content/SAM2-UNet/test_bgm.py:35: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  res = F.upsample(res, size=gt.shape, mode='bilinear', align_corners=False)
/content/SAM2-UNet/test_bgm.py:35: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  res = F.upsample(res, size=gt.shape, mode='bilinear', align_corners=False)
/content/SAM2-UNet/test_bgm.py:35: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  res = F.upsample(res, size=gt.shape, mode='bi

## 9. Evaluate — tính metrics

In [ ]:
%cd /content/SAM2-UNet
!sed -i "s/MSIOU = py_sod_metrics.MSIoU()/MSIOU = py_sod_metrics.MSIoU(with_dynamic=True, with_adaptive=True, with_binary=True)/g" eval.py
print('eval.py patched')

/content/SAM2-UNet
eval.py patched


In [ ]:
%%bash
cd /content/SAM2-UNet

for DATASET in Kvasir CVC-ClinicDB CVC-ColonDB CVC-300 ETIS-LaribPolypDB
do
  echo '============================'
  echo "Evaluating $DATASET"
  echo '============================'
  python eval.py \
    --dataset_name $DATASET \
    --pred_path /content/preds/sam2unet_bgm/$DATASET/ \
    --gt_path   /content/data/Polyp/TestDataset/$DATASET/masks/
done

Evaluating Kvasir
[0] Processing cju0u82z3cuma0835wlxrnrjv.png...
[1] Processing cju15wdt3zla10801odjiw7sy.png...
[2] Processing cju16ach3m1da0993r1dq3sn2.png...
[3] Processing cju16whaj0e7n0855q7b6cjkm.png...
[4] Processing cju17z0qongpa0993de4boim4.png...
[5] Processing cju1amqw6p8pw0993d9gc5crl.png...
[6] Processing cju1bm8063nmh07996rsjjemq.png...
[7] Processing cju1c3218411b08014g9f6gig.png...
[8] Processing cju1cbokpuiw70988j4lq1fpi.png...
[9] Processing cju1cj3f0qi5n0993ut8f49rj.png...
[10] Processing cju1cqc7n4gpy0855jt246k68.png...
[11] Processing cju1ddr6p4k5z08780uuuzit2.png...
[12] Processing cju1f8w0t65en0799m9oacq0q.png...
[13] Processing cju1h89h6xbnx08352k2790o9.png...
[14] Processing cju1hp9i2xu8e0988u2dazk7m.png...
[15] Processing cju2hfqnmhisa0993gpleeldd.png...
[16] Processing cju2hjrqcvi2j0801bx1i6gxg.png...
[17] Processing cju2hos57llxm08359g92p6jj.png...
[18] Processing cju2hqt33lmra0988fr5ijv8j.png...
[19] Processing cju2lberzkdzm09938cl40pog.png...
[20] Process

/usr/local/lib/python3.12/dist-packages/py_sod_metrics/sod_metrics.py:35: UserWarning: This class will be removed in the future, please use FmeasureV2 instead!
  warnings.warn("This class will be removed in the future, please use FmeasureV2 instead!")
/usr/local/lib/python3.12/dist-packages/py_sod_metrics/sod_metrics.py:35: UserWarning: This class will be removed in the future, please use FmeasureV2 instead!
  warnings.warn("This class will be removed in the future, please use FmeasureV2 instead!")
/usr/local/lib/python3.12/dist-packages/py_sod_metrics/sod_metrics.py:35: UserWarning: This class will be removed in the future, please use FmeasureV2 instead!
  warnings.warn("This class will be removed in the future, please use FmeasureV2 instead!")
/usr/local/lib/python3.12/dist-packages/py_sod_metrics/sod_metrics.py:35: UserWarning: This class will be removed in the future, please use FmeasureV2 instead!
  warnings.warn("This class will be removed in the future, please use FmeasureV2 ins